## Example 1: Path utilities with expand_path() and filefind()

In [ ]:
from ipython_genutils.path import expand_path, filefind, ensure_dir_exists
import os
import tempfile

# Example 1: Path utilities with expand_path() and filefind()
print("Example 1: Path utility functions")

# Test path expansion
paths_to_expand = ['~', '~/Documents', '$HOME', '.']
print("Expanding paths:")
for path in paths_to_expand:
    try:
        expanded = expand_path(path)
        print(f"  '{path}' -> '{expanded}'")
    except Exception as e:
        print(f"  '{path}' -> Error: {e}")
        raise

# Test file finding in common directories
search_paths = ['/usr/bin', '/usr/local/bin', '/bin']
programs = ['python3', 'ls', 'cat']

print(f"\nFinding programs in {search_paths}:")
for program in programs:
    try:
        found_path = filefind(program, search_paths)
        if found_path:
            print(f"  {program}: {found_path}")
        else:
            print(f"  {program}: not found")
    except Exception as e:
        print(f"  {program}: Error - {e}")
        raise

# Test directory creation (inside a temporary directory that is removed afterwards)
with tempfile.TemporaryDirectory() as base_dir:
    test_dir = os.path.join(base_dir, 'test_ipython_genutils')
    print(f"\nEnsuring directory exists: {test_dir}")
    ensure_dir_exists(test_dir)
    print(f"Directory exists: {os.path.exists(test_dir)}")
    assert os.path.isdir(test_dir)
    # Calling it again on an existing directory is a no-op
    ensure_dir_exists(test_dir)

    # filefind() returns the absolute path of the first match in the search path
    target = os.path.join(base_dir, 'notes.txt')
    open(target, 'w').close()
    assert filefind('notes.txt', [base_dir]) == target
    try:
        filefind('missing.txt', [base_dir])
        raise AssertionError("expected OSError for a missing file")
    except OSError:
        pass

assert expand_path('~') == os.path.expanduser('~')
assert expand_path('$HOME') == os.path.expandvars('$HOME')

print("Path utilities completed successfully")

## Example 2: Text formatting with columnize() and dedent()

In [ ]:
from ipython_genutils.text import columnize, dedent, compute_item_matrix

# Example 2: Text formatting with columnize() and dedent()
print("Example 2: Text formatting utilities")

# Test columnize function with different data
data_sets = [
    ['apple', 'banana', 'cherry', 'date', 'elderberry', 'fig', 'grape'],
    ['alpha', 'beta', 'gamma', 'delta', 'epsilon', 'zeta', 'eta', 'theta'],
    ['file1.txt', 'document.pdf', 'image.png', 'data.csv', 'script.py']
]

print("Columnizing different datasets:")
for i, data in enumerate(data_sets, 1):
    try:
        # Test with different display widths
        width = 50
        result = columnize(data, displaywidth=width)
        lines = result.split('\n')
        print(f"  Dataset {i} ({len(data)} items, width={width}):")
        print(f"    Result: {len(lines)} lines")
        print(f"    Sample: {lines[0][:40]}..." if lines[0] else "    Empty result")
    except Exception as e:
        print(f"  Dataset {i}: Error - {e}")
        raise

# Test dedent function
indented_texts = [
    "    Hello World\n    This is indented\n    Multiple lines",
    "\t\tTab indented\n\t\tSecond line",
    "  Mixed\n    indentation\n  levels"
]

print(f"\nDedenting indented text:")
for i, text in enumerate(indented_texts, 1):
    try:
        dedented = dedent(text)
        original_lines = len(text.split('\n'))
        dedented_lines = len(dedented.split('\n'))
        print(f"  Text {i}: {original_lines} lines -> {dedented_lines} lines processed")
    except Exception as e:
        print(f"  Text {i}: Error - {e}")
        raise

# columnize() fills columns top to bottom
assert columnize(['a', 'b', 'c', 'd'], displaywidth=8) == 'a  c\nb  d\n'
assert columnize(data_sets[0], displaywidth=50) == 'apple   cherry  elderberry  grape\nbanana  date    fig       \n'
matrix, info = compute_item_matrix(['a', 'b', 'c', 'd', 'e'], displaywidth=8)
assert matrix == [['a', 'c', 'e'], ['b', 'd', None]]
assert info['rows_numbers'] == 2 and info['columns_numbers'] == 3

# This dedent() keeps the first line as is and dedents the lines after it,
# the way docstrings are written
assert dedent(indented_texts[0]) == '    Hello World\nThis is indented\nMultiple lines'
assert dedent(indented_texts[2]) == '  Mixed\n  indentation\nlevels'

## Example 3: Encoding utilities with getdefaultencoding() and get_stream_enc()

In [ ]:
from ipython_genutils.encoding import getdefaultencoding, get_stream_enc
import sys
import io

# Example 3: Encoding utilities with getdefaultencoding() and get_stream_enc()
print("Example 3: Encoding detection utilities")

# Test default encoding detection
print("System encoding information:")
try:
    default_enc = getdefaultencoding()
    print(f"  Default encoding: {default_enc}")
    print(f"  Encoding type: {type(default_enc).__name__}")
except Exception as e:
    print(f"  Default encoding error: {e}")
    raise

# Test stream encoding detection
streams_to_test = [
    ('sys.stdout', sys.stdout),
    ('sys.stderr', sys.stderr),
    ('sys.stdin', sys.stdin)
]

print(f"\nStream encoding detection:")
for stream_name, stream in streams_to_test:
    try:
        if hasattr(stream, 'encoding'):
            encoding = get_stream_enc(stream)
            print(f"  {stream_name}: {encoding}")
        else:
            print(f"  {stream_name}: No encoding attribute")
    except Exception as e:
        print(f"  {stream_name}: Error - {e}")
        raise

# Test with custom streams
print(f"\nCustom stream encoding:")
test_streams = [
    io.StringIO(),
    io.BytesIO(),
]

for i, stream in enumerate(test_streams):
    try:
        stream_type = type(stream).__name__
        if hasattr(stream, 'encoding'):
            encoding = get_stream_enc(stream)
            print(f"  {stream_type}: {encoding}")
        else:
            print(f"  {stream_type}: No encoding (binary stream)")
    except Exception as e:
        print(f"  {stream_type}: Error - {e}")
        raise

import codecs

# The default encoding is always a codec name Python knows
assert codecs.lookup(default_enc)
# get_stream_enc() returns the stream's encoding, or the default when it has none
assert get_stream_enc(io.TextIOWrapper(io.BytesIO(), encoding='latin-1')) == 'latin-1'
assert get_stream_enc(io.StringIO()) is None
assert get_stream_enc(io.BytesIO(), 'fallback') == 'fallback'

## Example 4: Temporary files and directories with NamedFileInTemporaryDirectory()

In [ ]:
from ipython_genutils.tempdir import NamedFileInTemporaryDirectory, TemporaryWorkingDirectory
import os

# Example 4: Temporary files and directories with NamedFileInTemporaryDirectory()
print("Example 4: Temporary files and directories")

with NamedFileInTemporaryDirectory('test_file.tmp') as temp_file:
    temp_path = temp_file.name
    print(f"  Temporary file: {temp_path}")
    print(f"  File exists: {os.path.exists(temp_path)}")
    assert os.path.basename(temp_path) == 'test_file.tmp' and os.path.exists(temp_path)

    # Write some data
    temp_file.write(b'Hello from temporary file!')
    temp_file.flush()

    file_size = os.path.getsize(temp_path)
    print(f"  File size: {file_size} bytes")
    assert file_size == len(b'Hello from temporary file!')

# Leaving the block removes the file and the directory that held it
assert not os.path.exists(temp_path) and not os.path.exists(os.path.dirname(temp_path))

# TemporaryWorkingDirectory changes into a new directory and changes back on exit
start_dir = os.getcwd()
with TemporaryWorkingDirectory() as work_dir:
    assert os.path.realpath(os.getcwd()) == os.path.realpath(work_dir)
assert os.getcwd() == start_dir and not os.path.exists(work_dir)

## Example 5: Import utilities and Python 3 compatibility

In [ ]:
from ipython_genutils.importstring import import_item
from ipython_genutils import py3compat
import ipython_genutils

# Example 5: Import utilities and Python 3 compatibility
print("Example 5: Import utilities and compatibility")

# Test import_item function
module_paths = [
    'os.path',
    'json.dumps', 
    'datetime.datetime',
    'collections.defaultdict',
    'sys.version_info'
]

print("Testing import_item with different module paths:")
for module_path in module_paths:
    try:
        imported = import_item(module_path)
        item_type = type(imported).__name__
        print(f"  '{module_path}': {item_type} imported successfully")
        
        # Show some details about the imported item
        if hasattr(imported, '__name__'):
            print(f"    Name: {imported.__name__}")
        elif hasattr(imported, '__class__'):
            print(f"    Class: {imported.__class__.__name__}")
            
    except Exception as e:
        print(f"  '{module_path}': Error - {e}")
        raise

# Test py3compat module
print(f"\nTesting py3compat utilities:")
try:
    py3compat_attrs = [attr for attr in dir(py3compat) if not attr.startswith('_')]
    print(f"  Available py3compat functions: {len(py3compat_attrs)}")
    print(f"  Sample functions: {py3compat_attrs[:5]}")
    
    # Test string types if available
    if hasattr(py3compat, 'string_types'):
        string_types = py3compat.string_types
        print(f"  String types: {string_types}")
        
except Exception as e:
    print(f"  py3compat error: {e}")
    raise

# Show package information
print(f"\nPackage information:")
print(f"  ipython_genutils version: {ipython_genutils.version_info}")

import json, datetime, collections, os

assert import_item('json.dumps') is json.dumps
assert import_item('os.path') is os.path
assert import_item('datetime.datetime') is datetime.datetime
assert import_item('collections.defaultdict') is collections.defaultdict
try:
    import_item('json.not_a_name')
    raise AssertionError("expected ImportError")
except ImportError:
    pass

assert py3compat.string_types == (str,)
assert py3compat.cast_unicode(b'caf\xc3\xa9') == 'café'
assert py3compat.cast_bytes('café') == b'caf\xc3\xa9'
assert ipython_genutils.version_info[:2] >= (0, 2)